In [ ]:
'''CASMI26 | Two Answers, One List

Rank-only ensemble for two independently generated submissions.
Attach the V17 and V20 notebook outputs as Kaggle inputs, plus the competition
data and an offline RDKit wheel. It writes submission.csv.
'''


In [ ]:
'''CASMI26 | Inputs

Keep INPUTS empty to discover exactly two attached submission CSV files.
If Kaggle finds more, paste the two printed paths here.
'''
from pathlib import Path
import glob
import subprocess
import sys

INPUTS = []
TOP_N = 25
RRF_K = 60

wheel = sorted(glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True))
if wheel:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', wheel[0]], check=True)

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
import pandas as pd

def find_competition_file(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(f'Missing competition file: {name}')
    return Path(hits[0])

sample_path = find_competition_file('sample_submission.csv')
sample = pd.read_csv(sample_path)

def submission_candidates():
    found = []
    for path in sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True)):
        if Path(path).resolve() == sample_path.resolve():
            continue
        try:
            columns = pd.read_csv(path, nrows=1).columns.tolist()
        except Exception:
            continue
        if columns == ['molecule_id', 'smiles']:
            found.append(path)
    return found

paths = INPUTS or submission_candidates()
print('Candidate submissions:', *paths, sep='\n  ')
assert len(paths) == 2, 'Attach exactly two notebook outputs or set INPUTS to their submission.csv paths.'


In [ ]:
'''CASMI26 | Reciprocal Rank Fusion

A candidate receives 1 / (60 + rank) from each source.
Chemical duplicates are merged by tautomer-normalized InChIKey14.
'''
tautomer = rdMolStandardize.TautomerEnumerator()

def structure_key(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToInchiKey(tautomer.Canonicalize(mol)).split('-')[0]

def read_submission(path):
    frame = pd.read_csv(path, dtype={'molecule_id': str, 'smiles': str})
    assert frame.columns.tolist() == ['molecule_id', 'smiles'], f'Unexpected columns in {path}: {frame.columns.tolist()}'
    assert frame.molecule_id.is_unique and frame.smiles.notna().all(), f'Invalid rows in {path}'
    return frame.set_index('molecule_id').reindex(sample.molecule_id.astype(str))

sources = [read_submission(path) for path in paths]
assert all(source.smiles.notna().all() for source in sources), 'Each source must cover all competition molecule_id values.'

def fuse(rows):
    scores, smiles_by_key, backup = {}, {}, []
    for row in rows:
        for rank, smiles in enumerate(str(row).split(';')[:TOP_N], start=1):
            key = structure_key(smiles)
            if key is None:
                continue
            backup.append(smiles)
            smiles_by_key.setdefault(key, smiles)
            scores[key] = scores.get(key, 0.0) + 1.0 / (RRF_K + rank)
    ordered = [smiles_by_key[key] for key in sorted(scores, key=lambda key: (-scores[key], smiles_by_key[key]))]
    if len(ordered) < TOP_N:
        ordered.extend(backup)
    ordered.extend(['CCO'] * TOP_N)
    return ';'.join(ordered[:TOP_N]), len(scores)

def top25_set(value):
    return {structure_key(s) for s in str(value).split(';')[:TOP_N] if structure_key(s) is not None}

overlap = []
for left, right in zip(sources[0].smiles, sources[1].smiles):
    a, b = top25_set(left), top25_set(right)
    overlap.append(len(a & b) / max(1, len(a | b)))
print(f'Mean top-25 structure overlap: {sum(overlap) / len(overlap):.3f}')
if sum(overlap) / len(overlap) > 0.90:
    print('Warning: the two sources are very similar; RRF may not improve leaderboard score.')


In [ ]:
'''CASMI26 | Submission

Fuse both rank lists, retain sample_submission order, and audit the final CSV.
'''
fused = [fuse(rows) for rows in zip(sources[0].smiles, sources[1].smiles)]
submission = sample[['molecule_id']].copy()
submission['smiles'] = [value for value, _ in fused]

assert submission.molecule_id.equals(sample.molecule_id)
assert submission.molecule_id.is_unique
assert submission.smiles.notna().all()
assert submission.smiles.map(lambda value: len(value.split(';'))).eq(TOP_N).all()

submission.to_csv('submission.csv', index=False)
pd.DataFrame({'molecule_id': submission.molecule_id, 'unique_candidates': [count for _, count in fused]}).to_csv('diagnostics.csv', index=False)
print('Wrote submission.csv:', submission.shape)
display(submission.head())
